In [5]:
import pandas as pd
import numpy as np
import os
import copy

START_MINUTE = 8 * 60 + 33
LIMIT = 777
MEAL_TRIGGER = 200
MEAL_TIME = 30
DINNER_TRIGGER = 500
DINNER_DUR = 30
DINNER_END_LIMIT = 680
SHOP_TIME = 15
CHECKIN_TIME = 10
SHOW_DUR_MEAN = 15
SHOW_DUR_STD = 3
MEETGREET_DUR_MEAN = 15
MEETGREET_DUR_STD = 3
MAX_WAIT = 15
WALK_SPEED_MIN_PER_KM = 15
ZONE_IN_OUT_EXTRA = 3

fixed_zone_order = [
    "米奇大街","奇想花园","明日世界","梦幻世界",
    "宝藏湾","皮克斯玩具总动员","探险岛"
]
zone_list = fixed_zone_order.copy()

dist_matrix_km = np.array([
    [0.00,0.09,0.35,0.10,0.44,0.44,0.54],
    [0.09,0.00,0.28,0.17,0.53,0.52,0.60],
    [0.35,0.28,0.00,0.36,0.76,0.63,0.64],
    [0.10,0.17,0.36,0.00,0.40,0.35,0.44],
    [0.44,0.53,0.76,0.40,0.00,0.31,0.47],
    [0.44,0.52,0.63,0.35,0.31,0.00,0.17],
    [0.54,0.60,0.64,0.44,0.47,0.17,0.00]
])
zone2idx = {z:i for i,z in enumerate(zone_list)}

def get_walk_time(from_zone, to_zone):
    if from_zone not in zone2idx or to_zone not in zone2idx:
        return 0
    return int(round(dist_matrix_km[zone2idx[from_zone], zone2idx[to_zone]] * WALK_SPEED_MIN_PER_KM))

shows_raw = pd.read_excel("演出.xlsx", header=0)
shows_raw.columns = ["园区","演出","评分","场次"]
shows_raw["园区"] = shows_raw["园区"].ffill()

def parse_times(s):
    if pd.isna(s):
        return []
    s = str(s).replace("：",":").replace("，",",").replace("\u3001",",")
    out = []
    for t in s.split(","):
        t = t.strip()
        if not t:
            continue
        pm = "下午" in t
        t2 = t.replace("上午","").replace("下午","").strip()
        try:
            h,m = map(int, t2.split(":"))
        except:
            continue
        if pm and h<12:
            h+=12
        off = h*60+m - START_MINUTE
        if off>=0:
            out.append(off)
    return sorted(out)

zone_shows = {}
for _,row in shows_raw.iterrows():
    z = row["园区"]
    sc = pd.to_numeric(row["评分"], errors="coerce")
    if pd.isna(sc):
        sc=0.0
    zone_shows.setdefault(z,[]).append((row["演出"], float(sc), parse_times(row["场次"])))

def sample_dur():
    return max(5, int(round(np.random.normal(SHOW_DUR_MEAN, SHOW_DUR_STD))))

def sample_meetgreet_dur():
    return max(5, int(round(np.random.normal(MEETGREET_DUR_MEAN, MEETGREET_DUR_STD))))

def pick_show(zone, finish_offset):
    if zone not in zone_shows:
        return None
    best = None
    best_key = (MAX_WAIT+1, -99.0)
    for (name, sc, slots) in zone_shows[zone]:
        if not slots:
            continue
        idx = int(np.searchsorted(slots, finish_offset))
        if idx>=len(slots):
            continue
        slot = slots[idx]
        wait = slot - finish_offset
        if wait > MAX_WAIT:
            continue
        key = (wait, -sc)
        if key < best_key:
            best_key = key
            best = (name, sc, slot, wait, sample_dur())
    return best

def parse_meetgreet_time_range(s):
    if pd.isna(s):
        return None,None
    s = str(s).replace("：",":").replace("；",":").replace(" ","")
    parts = s.split("-")
    if len(parts)!=2:
        return None,None
    try:
        start_h,start_m = map(int, parts[0].split(":"))
        end_h,end_m = map(int, parts[1].split(":"))
        start_off = start_h*60+start_m - START_MINUTE
        end_off = end_h*60+end_m - START_MINUTE
        return max(0,start_off), end_off
    except:
        return None,None

meetgreet_raw = pd.read_excel("见面会.xlsx", header=0)
meetgreet_raw.columns = ["见面会名称","地点","开放时间","评分","排队时间"]
meetgreets = []
for _,row in meetgreet_raw.iterrows():
    name = row["见面会名称"]
    zone = row["地点"]
    score = pd.to_numeric(row["评分"], errors="coerce")
    queue = pd.to_numeric(row["排队时间"], errors="coerce")
    if pd.isna(score) or pd.isna(queue):
        continue
    start_off,end_off = parse_meetgreet_time_range(row["开放时间"])
    if start_off is None or end_off is None:
        continue
    meetgreets.append({
        "名称":name,"地点":zone,"评分":float(score),"排队时间":int(queue),
        "开始时间":start_off,"结束时间":end_off
    })

shops = pd.read_excel("商店.xlsx")
shops.columns = ["园区","商店","评分"]
shops["园区"] = shops["园区"].ffill()
shops = shops.dropna(subset=["商店","评分"])
shops["评分"] = pd.to_numeric(shops["评分"], errors="coerce")
shops = shops.dropna(subset=["评分"])
best_shop = shops.sort_values("评分", ascending=False).groupby("园区").first()
shop_score_map = {z: float(best_shop.loc[z,"评分"]) for z in best_shop.index}
shop_name_map = {z: best_shop.loc[z,"商店"] for z in best_shop.index}

def find_queue_file_for_ride(ride_name):
    for f in os.listdir("."):
        if f.endswith(".xlsx") and ride_name in f:
            return f
    return None

def load_queue_data(ride_name, scenario_name):
    file_path = find_queue_file_for_ride(ride_name)
    if file_path is None:
        return None
    sheet_candidates = {
        "周中普通": ["4月23日", "2026-04-23", "4月23日 星期四"],
        "周末普通": ["4月25日", "2026-04-25", "4月25日 星期六"],
        "节假日普通": ["5月1日", "2026-05-01", "5月1日 星期五"]
    }
    candidates = sheet_candidates.get(scenario_name, [])
    xl = pd.ExcelFile(file_path)
    sheet_name = None
    for cand in candidates:
        if cand in xl.sheet_names:
            sheet_name = cand
            break
    if sheet_name is None:
        sheet_name = xl.sheet_names[0]
    try:
        df = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
    except Exception:
        return None
    time_keywords = ["时间", "时间点", "时间段", "Time"]
    header_row_idx = None
    for i, row in df.iterrows():
        first_cell = str(row.iloc[0]).strip()
        if any(kw in first_cell for kw in time_keywords):
            header_row_idx = i
            break
    if header_row_idx is not None:
        df = pd.read_excel(file_path, sheet_name=sheet_name, header=header_row_idx)
    else:
        df = pd.read_excel(file_path, sheet_name=sheet_name, header=0)
    time_col = None
    for col in df.columns:
        col_str = str(col).lower()
        if "时间" in col_str or "time" in col_str:
            time_col = col
            break
    if time_col is None:
        time_col = df.columns[0]
    hour_values = {}
    for _, row in df.iterrows():
        time_val = row[time_col]
        if pd.isna(time_val):
            continue
        try:
            time_str = str(time_val).strip()
            if ':' in time_str:
                h = int(time_str.split(':')[0])
            else:
                h = int(time_str)
        except:
            continue
        values = []
        for col in df.columns:
            if col == time_col:
                continue
            v = row[col]
            if pd.notna(v):
                try:
                    values.append(float(v))
                except:
                    pass
        if values:
            hour_values.setdefault(h, []).extend(values)
    if not hour_values:
        return None
    hour_avg = {h: np.mean(vals) for h, vals in hour_values.items()}
    time_points = []
    queue_values = []
    for hour in range(8, 22):
        for minute in range(0, 60, 10):
            abs_min = hour*60 + minute
            time_points.append(abs_min)
            q = hour_avg.get(hour)
            if q is None:
                nearest = min(hour_avg.keys(), key=lambda x: abs(x-hour))
                q = hour_avg[nearest]
            queue_values.append(q)
    def get_queue(current_abs):
        if current_abs < time_points[0]:
            return queue_values[0]
        if current_abs > time_points[-1] + 10:
            return queue_values[-1]
        idx = int((current_abs - time_points[0]) // 10)
        idx = max(0, min(idx, len(queue_values)-1))
        return queue_values[idx]
    return get_queue

def simulate_sequence(proj_indices, items, meetgreet, meetgreet_insert):
    t = 0.0
    visited_zones = set()
    last_zone = None
    lunch_done = False
    dinner_done = False
    show_list = []
    shop_score = 0.0
    mg_score = 0.0
    mg_used = False
    actual_score = 0.0
    for idx in proj_indices:
        item = items[idx]
        zone = item["园区"]
        new_zone = zone not in visited_zones
        if new_zone and last_zone is not None:
            walk = get_walk_time(last_zone, zone) + ZONE_IN_OUT_EXTRA
            t += walk
        if new_zone:
            t += CHECKIN_TIME
        if item["动态"] and item["实时排队函数"] is not None and item["历史平均排队"] is not None:
            abs_time = START_MINUTE + t
            t_app = item["实时排队函数"](abs_time)
            t_bar = item["历史平均排队"]
            if t_app > 0 and t_bar > 0:
                factor = max(1.0, np.sqrt(t_bar / t_app))
                actual_score += item["原始评分"] * factor
            else:
                actual_score += 0.0
            t += t_app
        else:
            actual_score += item["原始评分"]
        t += item["游玩时间"]
        if new_zone:
            show_info = pick_show(zone, t)
            if show_info is not None:
                s_name, s_sc, s_slot, s_wait, s_dur = show_info
                if t + s_wait + s_dur + SHOP_TIME + CHECKIN_TIME + (MEAL_TIME if not lunch_done and t > MEAL_TRIGGER else 0) + (DINNER_DUR if not dinner_done and t > DINNER_TRIGGER and t+DINNER_DUR <= DINNER_END_LIMIT else 0) <= LIMIT:
                    t += s_wait + s_dur
                    show_list.append({"园区":zone,"演出":s_name,"评分":s_sc,"slot_off":s_slot,"wait":s_wait,"dur":s_dur})
                    actual_score += s_sc
        if new_zone:
            shop_score += shop_score_map.get(zone, 0)
            t += SHOP_TIME
        if meetgreet and meetgreet_insert and not mg_used:
            insert_zone_idx, insert_time_after = meetgreet_insert
            if new_zone and len(visited_zones) == insert_zone_idx and zone == fixed_zone_order[insert_zone_idx]:
                participate_time = t
                if meetgreet["开始时间"] <= participate_time <= meetgreet["结束时间"]:
                    t += meetgreet["排队时间"] + meetgreet["duration"]
                    mg_score = meetgreet["评分"]
                    mg_used = True
        if not lunch_done and t > MEAL_TRIGGER:
            t += MEAL_TIME
            lunch_done = True
        if not dinner_done and t > DINNER_TRIGGER:
            if START_MINUTE + t + DINNER_DUR <= START_MINUTE + DINNER_END_LIMIT:
                t += DINNER_DUR
                dinner_done = True
        if new_zone:
            visited_zones.add(zone)
            last_zone = zone
    total_score_actual = actual_score + shop_score + mg_score
    return t, total_score_actual, show_list, shop_score, mg_score

def greedy_refine(initial_indices, items, meetgreet, meetgreet_insert, max_iter=100):
    current_indices = list(initial_indices)
    current_time, current_score, _, _, _ = simulate_sequence(current_indices, items, meetgreet, meetgreet_insert)
    improved = True
    iteration = 0
    total_items = len(items)
    raw_scores = [it["原始评分"] for it in items]
    while improved and iteration < max_iter:
        improved = False
        iteration += 1
        for i in range(len(current_indices)):
            new_seq = current_indices[:i] + current_indices[i+1:]
            new_time, new_score, _, _, _ = simulate_sequence(new_seq, items, meetgreet, meetgreet_insert)
            if new_time <= LIMIT and new_score > current_score + 1e-6:
                current_indices = new_seq
                current_score = new_score
                current_time = new_time
                improved = True
                break
        if improved:
            continue
        selected_set = set(current_indices)
        candidates = sorted([i for i in range(total_items) if i not in selected_set], key=lambda i: raw_scores[i], reverse=True)
        for add_idx in candidates:
            insert_pos = 0
            while insert_pos < len(current_indices) and current_indices[insert_pos] < add_idx:
                insert_pos += 1
            new_seq = current_indices[:insert_pos] + [add_idx] + current_indices[insert_pos:]
            new_time, new_score, _, _, _ = simulate_sequence(new_seq, items, meetgreet, meetgreet_insert)
            if new_time <= LIMIT and new_score > current_score + 1e-6:
                current_indices = new_seq
                current_score = new_score
                current_time = new_time
                improved = True
                break
        if improved:
            continue
        for i in range(len(current_indices)):
            for add_idx in candidates:
                if add_idx == current_indices[i]:
                    continue
                temp = current_indices[:i] + current_indices[i+1:]
                insert_pos = 0
                while insert_pos < len(temp) and temp[insert_pos] < add_idx:
                    insert_pos += 1
                new_seq = temp[:insert_pos] + [add_idx] + temp[insert_pos:]
                new_time, new_score, _, _, _ = simulate_sequence(new_seq, items, meetgreet, meetgreet_insert)
                if new_time <= LIMIT and new_score > current_score + 1e-6:
                    current_indices = new_seq
                    current_score = new_score
                    current_time = new_time
                    improved = True
                    break
            if improved:
                break
    return current_indices, current_score, current_time

def run_scenario(project_csv_path, scenario_name, reset_seed=True):
    if reset_seed:
        np.random.seed(42)
    projects = pd.read_csv(project_csv_path)
    projects.columns = ["园区","项目","评分","时间"]
    projects["园区"] = projects["园区"].ffill()
    projects = projects.dropna(subset=["项目","评分","时间"])
    projects["评分"] = pd.to_numeric(projects["评分"], errors="coerce")
    projects["时间"] = pd.to_numeric(projects["时间"], errors="coerce")
    projects = projects.dropna(subset=["评分","时间"])
    items = []
    for z in fixed_zone_order:
        zp = projects[projects["园区"] == z].sort_values(["评分","时间"], ascending=[False,True])
        for _, row in zp.iterrows():
            name = row["项目"]
            orig_score = float(row["评分"])
            time_col_value = float(row["时间"])
            queue_func = load_queue_data(name, scenario_name)
            is_dynamic = (queue_func is not None)
            if is_dynamic:
                t_bar = time_col_value
                ride_duration = 0
            else:
                t_bar = None
                ride_duration = time_col_value
            items.append({
                "园区": z, "名称": name,
                "原始评分": orig_score, "游玩时间": ride_duration,
                "动态": is_dynamic,
                "历史平均排队": t_bar,
                "实时排队函数": queue_func if is_dynamic else None
            })
    n = len(items)
    scores_arr = np.array([x["原始评分"] for x in items], dtype=np.float64)
    suffix_score = np.zeros(n+1)
    for i in range(n-1,-1,-1):
        suffix_score[i] = suffix_score[i+1] + scores_arr[i]
    best_score_g = -1.0
    best_choice_g = []
    best_shows_g = []
    best_meetgreet_g = None
    best_meetgreet_insert_g = None
    best_total_time_g = 0
    def dfs(i, chosen, shows, visited, proj_time, proj_score, show_score,
            meetgreet_data, meetgreet_inserted_at, lunch_done, dinner_done):
        nonlocal best_score_g, best_choice_g, best_shows_g, best_meetgreet_g, best_meetgreet_insert_g, best_total_time_g
        shop_sc = sum(shop_score_map.get(z,0) for z in visited)
        mg_sc = meetgreet_data["评分"] if (meetgreet_data and meetgreet_inserted_at) else 0
        optimistic = proj_score + show_score + shop_sc + mg_sc + suffix_score[i]
        if optimistic <= best_score_g:
            return
        shop_t = len(visited) * SHOP_TIME
        chk_t = len(visited) * CHECKIN_TIME
        mg_t = (meetgreet_data["排队时间"] + meetgreet_data["duration"]) if (meetgreet_data and meetgreet_inserted_at) else 0
        lunch_t = 0
        new_lunch_done = lunch_done
        if not lunch_done and proj_time > MEAL_TRIGGER:
            lunch_t = MEAL_TIME
            new_lunch_done = True
        dinner_t = 0
        new_dinner_done = dinner_done
        if not dinner_done and proj_time > DINNER_TRIGGER:
            if proj_time + DINNER_DUR <= DINNER_END_LIMIT:
                dinner_t = DINNER_DUR
                new_dinner_done = True
        total_t = proj_time + shop_t + chk_t + mg_t + lunch_t + dinner_t
        if total_t > LIMIT:
            return
        if i == n:
            sc = proj_score + show_score + shop_sc + mg_sc
            if sc > best_score_g:
                best_score_g = sc
                best_choice_g = chosen.copy()
                best_shows_g = shows.copy()
                best_meetgreet_g = meetgreet_data.copy() if meetgreet_data else None
                best_meetgreet_insert_g = meetgreet_inserted_at
                best_total_time_g = total_t
            return
        chosen.append(i)
        item = items[i]
        zone = item["园区"]
        added = zone not in visited
        if added:
            visited.add(zone)
        dynamic_wait = 0
        actual_score = item["原始评分"]
        if item["动态"] and item["实时排队函数"] is not None and item["历史平均排队"] is not None:
            abs_time = START_MINUTE + proj_time
            t_app = item["实时排队函数"](abs_time)
            t_bar = item["历史平均排队"]
            dynamic_wait = t_app
            if t_app > 0 and t_bar > 0:
                ratio = t_bar / t_app
                factor = max(1.0, np.sqrt(ratio))
                actual_score = item["原始评分"] * factor
            else:
                actual_score = 0.0
        else:
            dynamic_wait = 0
        project_time_inc = dynamic_wait + item["游玩时间"]
        new_pt = proj_time + project_time_inc
        new_ps = proj_score + actual_score
        show_appended = False
        extra_show_t = 0
        extra_show_s = 0.0
        if added:
            info = pick_show(zone, new_pt)
            if info is not None:
                s_name, s_sc, s_slot, s_wait, s_dur = info
                candidate_t = new_pt + s_wait + s_dur
                st = len(visited) * SHOP_TIME
                ct = len(visited) * CHECKIN_TIME
                lunch_tmp = MEAL_TIME if not lunch_done and new_pt > MEAL_TRIGGER else 0
                dinner_tmp = DINNER_DUR if (not dinner_done and new_pt > DINNER_TRIGGER and new_pt+DINNER_DUR<=DINNER_END_LIMIT) else 0
                mg_tmp = (meetgreet_data["排队时间"]+meetgreet_data["duration"]) if (meetgreet_data and meetgreet_inserted_at) else 0
                if candidate_t + st + ct + lunch_tmp + dinner_tmp + mg_tmp <= LIMIT:
                    shows.append({"园区":zone,"演出":s_name,"评分":s_sc,"slot_off":s_slot,"wait":s_wait,"dur":s_dur})
                    show_appended = True
                    extra_show_t = s_wait + s_dur
                    extra_show_s = s_sc
        if meetgreet_data and not meetgreet_inserted_at and added:
            zone_idx = len(visited) - 1
            mg_zone = meetgreet_data["地点"]
            valid_zones = set()
            if zone_idx < len(fixed_zone_order):
                valid_zones.add(fixed_zone_order[zone_idx])
            if zone_idx+1 < len(fixed_zone_order):
                valid_zones.add(fixed_zone_order[zone_idx+1])
            if mg_zone in valid_zones:
                mg_queue = meetgreet_data["排队时间"]
                mg_dur = meetgreet_data["duration"]
                participate_time = new_pt + extra_show_t + mg_queue
                if meetgreet_data["开始时间"] <= participate_time <= meetgreet_data["结束时间"]:
                    mg_total = mg_queue + mg_dur
                    new_time_with_mg = new_pt + extra_show_t + mg_total
                    st = len(visited) * SHOP_TIME
                    ct = len(visited) * CHECKIN_TIME
                    lunch_tmp = MEAL_TIME if not lunch_done and new_time_with_mg > MEAL_TRIGGER else 0
                    dinner_tmp = DINNER_DUR if (not dinner_done and new_time_with_mg > DINNER_TRIGGER and new_time_with_mg+DINNER_DUR<=DINNER_END_LIMIT) else 0
                    if new_time_with_mg + st + ct + lunch_tmp + dinner_tmp <= LIMIT:
                        dfs(i+1, chosen, shows, visited,
                            new_time_with_mg, new_ps, show_score+extra_show_s,
                            meetgreet_data, (zone_idx, new_pt+extra_show_t),
                            new_lunch_done, new_dinner_done)
        dfs(i+1, chosen, shows, visited,
            new_pt + extra_show_t, new_ps, show_score+extra_show_s,
            meetgreet_data, meetgreet_inserted_at,
            new_lunch_done, new_dinner_done)
        if show_appended:
            shows.pop()
        if added:
            visited.remove(zone)
        chosen.pop()
        dfs(i+1, chosen, shows, visited, proj_time, proj_score, show_score,
            meetgreet_data, meetgreet_inserted_at, lunch_done, dinner_done)
    dfs(0, [], [], set(), 0, 0.0, 0.0, None, None, False, False)
    for mg in meetgreets:
        mg_with_dur = mg.copy()
        mg_with_dur["duration"] = sample_meetgreet_dur()
        dfs(0, [], [], set(), 0, 0.0, 0.0, mg_with_dur, None, False, False)
    refined_indices, refined_score, refined_time = greedy_refine(
        best_choice_g, items, best_meetgreet_g, best_meetgreet_insert_g, max_iter=50)
    if refined_score > best_score_g + 1e-6:
        best_score_g = refined_score
        best_choice_g = refined_indices
        best_total_time_g = refined_time
        _, _, new_shows, _, _ = simulate_sequence(best_choice_g, items, best_meetgreet_g, best_meetgreet_insert_g)
        best_shows_g = new_shows
    return {
        "scenario": scenario_name,
        "best_score": best_score_g,
        "best_choice": best_choice_g.copy(),
        "best_shows": best_shows_g.copy(),
        "best_meetgreet": best_meetgreet_g.copy() if best_meetgreet_g else None,
        "best_meetgreet_insert": best_meetgreet_insert_g,
        "total_time": best_total_time_g,
        "items": items
    }

scenarios = [
    ("项目_周中普通.csv", "周中普通"),
    ("项目_节假日普通.csv", "节假日普通"),
    ("项目_周末普通.csv", "周末普通")
]

results = []
for csv_file, name in scenarios:
    res = run_scenario(csv_file, name, reset_seed=True)
    results.append(res)

print("\n" + "=" * 80)
print("                           对比表")
print("=" * 80)
print(f"{'场景':<12} {'最优总满意度':<16} {'游玩项目数':<12}")
print("-" * 80)
for res in results:
    project_count = len(res["best_choice"])
    print(f"{res['scenario']:<12} {res['best_score']:<16.2f} {project_count:<12}")
print("=" * 80)


                           对比表
场景           最优总满意度           游玩项目数       
--------------------------------------------------------------------------------
周中普通         151.05           23          
周末普通         135.56           18          
节假日普通        132.91           20          
